# Create and verify a small dataset

Build a deterministic CSV dataset, calculate basic statistics, write a Data Card, and produce a SHA-256 manifest before publishing. This local tutorial uses only Python's standard library.

In [ ]:
import csv
import hashlib
import json
from pathlib import Path
from statistics import mean

OUTPUT = Path("superii-dataset-tutorial")
OUTPUT.mkdir(exist_ok=True)

## Define transparent source records

These rows are synthetic tutorial data. A real Data Card must document collection, consent, licensing, exclusions, known bias, and intended use.

In [ ]:
rows = [
    {"sample_id": "sii-001", "text": "clear documentation", "quality": 5},
    {"sample_id": "sii-002", "text": "reproducible example", "quality": 4},
    {"sample_id": "sii-003", "text": "bounded preview", "quality": 5},
]
csv_path = OUTPUT / "train.csv"
with csv_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
print(csv_path.read_text(encoding="utf-8"))

In [ ]:
stats = {
    "rows": len(rows),
    "columns": list(rows[0]),
    "unique_sample_ids": len({row['sample_id'] for row in rows}),
    "mean_quality": mean(row["quality"] for row in rows),
    "missing_values": sum(value in (None, "") for row in rows for value in row.values()),
}
assert stats["unique_sample_ids"] == stats["rows"]
assert stats["missing_values"] == 0
print(json.dumps(stats, indent=2))

## Write the Data Card and immutable manifest

The manifest binds the published file path, byte size, media type, and checksum. Recalculate it after any content change.

In [ ]:
data_card = """# Tutorial quality phrases

## Summary
Three synthetic English phrases created solely for this Super ii tutorial.

## License
CC0-1.0 for these synthetic rows.

## Intended use
Demonstrating local dataset validation. Not suitable for training or evaluation claims.

## Provenance and limitations
Created manually for this notebook; no people, private data, or external sources are represented.
"""
(OUTPUT / "README.md").write_text(data_card, encoding="utf-8")

def file_record(path):
    content = path.read_bytes()
    return {
        "path": path.name,
        "size_bytes": len(content),
        "sha256": hashlib.sha256(content).hexdigest(),
    }

manifest = [file_record(OUTPUT / name) for name in ("README.md", "train.csv")]
(OUTPUT / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))

In [ ]:
for record in manifest:
    path = OUTPUT / record["path"]
    assert path.stat().st_size == record["size_bytes"]
    assert hashlib.sha256(path.read_bytes()).hexdigest() == record["sha256"]
print("Verified every manifest entry. Review the files before uploading them.")